In [1]:
import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
from pymc.math import dot, invlogit

rng = np.random.default_rng(6420)

In [2]:
data_frame = pd.read_csv('data_clean/bfy_age4_clean.csv')

In [3]:
# Standardize helper:

def standardize(dataframe):
    continuous = [col for col in dataframe if dataframe[col].nunique() > 2]
    standardized = dataframe.copy()

    standardized[continuous] = (dataframe[continuous] - dataframe[continuous].mean()) / (2 * dataframe[continuous].std())

    return standardized

In [4]:
covariates = ['birthweight_oz', 'gest_age', 'mom_age', 'mom_health', 'cesd', 'n_children', 'n_adults', 'hh_income', 'net_worth', 'female', 'smoked', 'alcohol', 'biodad_in_hh', 'spanish', 'under4']

In [5]:
df_standard = standardize(data_frame[covariates])
covs = df_standard.to_numpy(copy=True)

missing = np.isnan(covs)
covs[np.isnan(covs)] = 0

In [6]:
categorical_vars = ['site', 'mom_educ', 'mom_race', 'marital', 'assessor']

parts = [covs, missing]
for elem in categorical_vars: 
    x = data_frame[elem].to_numpy(copy=True)
    x[np.isnan(x)] = 99
    parts.append(pd.get_dummies(x).to_numpy()[:, 1:])

matrix = np.concatenate(parts, axis=1)

matrix.shape

(1000, 67)

In [7]:
treat = data_frame['treat'].to_numpy(copy=True)

In [8]:
priors_def = ['skeptical', 'weak', 'enthusiastic']
prior_means = [0, 0, 0.20]
prior_sds = [0.10, 0.50, 0.10]

In [9]:
outcomes = ['vocab_z', 'ef_z', 'cbcl_z', 'eeg_z', 'matrices_z', 'reading_z']
signs = [1, 1, -1, 1, 1, 1]
len(priors_def), len(prior_means), len(prior_sds), len(outcomes), len(signs)

(3, 3, 3, 6, 6)

In [10]:
labels = []
summaries = []

for out, sign in zip(outcomes, signs):
    y = sign * data_frame[out].to_numpy(copy=True)
    missing_mask = np.isnan(y)
    y_obs = y[~missing_mask]
    treat_obs = treat[~missing_mask]
    matrix_obs = matrix[~missing_mask]

    varies_mask = matrix_obs.std(axis=0) > 0
    matrix_obs = matrix_obs[:, varies_mask]

    for name, prior_mean, prior_sd in zip(priors_def, prior_means, prior_sds):
        with pm.Model() as model:
            tau = pm.Gamma('tau', 0.001, 0.001)
            alpha = pm.Normal('intercept', 0, tau=0.001)
            delta = pm.Normal('delta', mu=prior_mean, sigma=prior_sd)
            beta = pm.Normal('beta', 0, tau=0.001, shape = matrix_obs.shape[1])
            variance = pm.Deterministic('variance', 1 / tau)

            prob_pos = pm.Deterministic('prob_positive', pm.math.switch(delta >=0, 1, 0))
            prob_small = pm.Deterministic('prob_small', pm.math.switch(delta >= 0.10, 1, 0))
            prob_expected = pm.Deterministic('prob_expected', pm.math.switch(delta >= 0.20, 1, 0))
            prob_negligible = pm.Deterministic('prob_negligible', pm.math.switch(pm.math.abs(delta) <= 0.10, 1, 0))

            mu = alpha + delta * treat_obs + pm.math.dot(matrix_obs, beta)
            pm.Normal('likelihood', mu=mu, tau=tau, observed=y_obs)

            trace = pm.sample(3000, random_seed=6420)

        labels.append(out + ' / ' + name)
        summaries.append(az.summary(trace, var_names=['delta', 'prob_positive', 'prob_small', 'prob_expected', 'prob_negligible'],ci_prob=0.95))

NUTS[nutpie]: [tau, intercept, delta, beta]


/Users/abtinkhashayar/pymc_env/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/Users/abtinkhashayar/pymc_env/lib/python3.12/site-packages/arviz_stats/base/diagnostics.py:90: RuntimeWarning: invalid value encountered in scalar divide
  (between_chain_variance / within_chain_variance + num_samples - 1) / (num_samples)
/Users/abtinkhashayar/pymc_env/lib/python3.12/site-packages/arviz_stats/base/diagnostics.py:313: RuntimeWarning: invalid value encountered in scalar divide
  varsd = varvar / evar / 4
NUTS[nutpie]: [tau, intercept, delta, beta]


/Users/abtinkhashayar/pymc_env/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

NUTS[nutpie]: [tau, intercept, delta, beta]


/Users/abtinkhashayar/pymc_env/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

... (26 outputs left out)

NUTS[nutpie]: [tau, intercept, delta, beta]


/Users/abtinkhashayar/pymc_env/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

NUTS[nutpie]: [tau, intercept, delta, beta]


/Users/abtinkhashayar/pymc_env/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

In [11]:
rows = []

for summary in summaries:
    means = summary['mean'].to_numpy()
    lower = summary['eti95_lb'].to_numpy()
    upper = summary['eti95_ub'].to_numpy()
    rows.append([means[0], lower[0], upper[0], means[1], means[2], means[3], means[4]])

tabel_out = pd.DataFrame(rows, columns=['effect', 'lower', 'upper', 'P(>0)', 'P(>0.10)', 'P(>0.20)', 'P(|effect|<0.10)'])
tabel_out['model'] = labels
tabel_out.set_index('model')

,effect,lower,upper,P(>0),P(>0.10),P(>0.20),P(|effect|<0.10)
model,,,,,,,
vocab_z / skeptical,-0.037940,-0.152410,0.078501,0.254583,0.010333,0.000000,0.847083
vocab_z / weak,-0.055801,-0.196082,0.088633,0.218000,0.017667,0.000333,0.712083
vocab_z / enthusiastic,0.030550,-0.083595,0.145805,0.702917,0.118250,0.001583,0.869500
ef_z / skeptical,0.037114,-0.079918,0.154416,0.734083,0.147167,0.003667,0.842250
ef_z / weak,0.056112,-0.086240,0.197067,0.780250,0.273167,0.022750,0.711000
ef_z / enthusiastic,0.107034,-0.009111,0.222577,0.963667,0.545333,0.058250,0.454667
cbcl_z / skeptical,-0.025115,-0.130299,0.081320,0.324500,0.009583,0.000000,0.903583
cbcl_z / weak,-0.037820,-0.167335,0.089316,0.283250,0.017500,0.000083,0.809917
cbcl_z / enthusiastic,0.035003,-0.071438,0.141969,0.734500,0.115250,0.001250,0.877417


In [12]:
y = data_frame['diagnosis'].to_numpy(copy=True)
missing_mask = np.isnan(y)
y_obs = y[~missing_mask]

treat_obs = treat[~missing_mask]
matrix_obs = matrix[~missing_mask]

matrix_obs = matrix_obs[:, :-17]

varies_mask = matrix_obs.std(axis=0) > 0
matrix_obs =  matrix_obs[:, varies_mask]

diag_labels = []
diag_summs = []

for name, prior_sd in zip(priors_def[:2], prior_sds[:2]):
    with pm.Model() as diag_model:
        alpha = pm.Normal('alpha', mu=0, sigma= 10)
        cohen_d = pm.Normal('cohen_d', mu=0, sigma=prior_sd)
        delta = pm.Deterministic('delta', cohen_d * np.pi / np.sqrt(3))
        betas = pm.Normal('beta', mu=0, sigma=5, shape=matrix_obs.shape[1])

        prob = invlogit(alpha + delta * treat_obs + dot(matrix_obs, betas))

        prob_positive = pm.Deterministic('prob_positive', pm.math.switch(cohen_d>= 0, 1, 0))
        prob_negligible = pm.Deterministic('prob_negligible', pm.math.switch(pm.math.abs(cohen_d) <=0.10, 1, 0))

        pm.Bernoulli('y', p=prob, observed=y_obs)

        trace = pm.sample(5000, random_seed=6420)

    diag_labels.append('diagnosis / ' + name)
    diag_summs.append(az.summary(trace, var_names=['cohen_d', 'prob_positive', 'prob_negligible'], ci_prob=0.95))

NUTS[nutpie]: [alpha, cohen_d, beta]


/Users/abtinkhashayar/pymc_env/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

NUTS[nutpie]: [alpha, cohen_d, beta]


/Users/abtinkhashayar/pymc_env/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

In [13]:
rows = []
for summary in diag_summs:
    means = summary['mean'].to_numpy()
    lower = summary['eti95_lb'].to_numpy()
    upper = summary['eti95_ub'].to_numpy()
    rows.append([means[0], lower[0], upper[0], means[1], means[2]])

diag_table = pd.DataFrame(rows, columns=['effect (d)', 'lower', 'upper', 'P(>0)', 'P(|effect|<0.10)'])
diag_table['model'] = diag_labels
diag_table.set_index('model')

,effect (d),lower,upper,P(>0),P(|effect|<0.10)
model,,,,,
diagnosis / skeptical,0.027144,-0.124360,0.179862,0.63475,0.77540
diagnosis / weak,0.065533,-0.169143,0.297862,0.71060,0.52765
